# 전세대출 Top 5 상품 추천

**실행 순서**: Cell 1 → 2 → 3 → 4(고객 ID 입력) → 5(추천 실행)  
**고객 변경**: Cell 4 의 `USER_ID` 만 바꾸고 Cell 5 를 다시 실행하면 됩니다.

### 필요 파일 (`/content/` 아래에 업로드)
| 파일 | 설명 |
|------|------|
| `jeonse_customer_profiles.csv` | 고객 개인·재무 정보 |
| `jeonse_property_gate.csv` | 매물·전세계약 정보 |
| `merge_jeonse_product_catalog.csv` | 전세대출 상품 카탈로그 (A1~A5 정부지원 + B001~ 금융권) |

In [ ]:
# ══════════════════════════════════════════════════════════════
# Cell 1 : 라이브러리 & 전역 상수
# ══════════════════════════════════════════════════════════════
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

PROFILES_PATH = '/content/jeonse_customer_profiles.csv'
GATE_PATH     = '/content/jeonse_property_gate.csv'
CATALOG_PATH  = '/content/merge_jeonse_product_catalog.csv'

LOAN_TERM_MONTHS = 24
GUARANTEE_SCORE  = {'HUG': 100, 'HF': 80, 'SGI': 60}

# (Approval, Interest, Limit, Affordability, Guarantee, Preference)
WEIGHT_PROFILES = {
    'default':           (0.25, 0.25, 0.20, 0.15, 0.10, 0.05),
    'rate_sensitive':    (0.20, 0.38, 0.18, 0.14, 0.05, 0.05),
    'limit_constrained': (0.20, 0.15, 0.38, 0.15, 0.07, 0.05),
    'credit_vulnerable': (0.40, 0.20, 0.18, 0.12, 0.05, 0.05),
    'youth':             (0.20, 0.35, 0.20, 0.10, 0.10, 0.05),
    'newlywed_newborn':  (0.22, 0.30, 0.25, 0.13, 0.05, 0.05),
    'high_dsr_risk':     (0.25, 0.18, 0.15, 0.32, 0.05, 0.05),
}
WEIGHT_LABELS = {
    'default':           '기본형',
    'rate_sensitive':    '금리민감형',
    'limit_constrained': '한도부족형',
    'credit_vulnerable': '신용취약형',
    'youth':             '청년무주택형',
    'newlywed_newborn':  '신혼·신생아형',
    'high_dsr_risk':     '상환위험형',
}
print('✓ Cell 1 완료')

In [ ]:
# ══════════════════════════════════════════════════════════════
# Cell 2 : 데이터 로드
# ══════════════════════════════════════════════════════════════
profiles = pd.read_csv(PROFILES_PATH, dtype={'user_id': str})
gates    = pd.read_csv(GATE_PATH,     dtype={'user_id': str})
catalog  = pd.read_csv(CATALOG_PATH)

num_cols = [
    'base_rate_min', 'base_rate_max',
    'deposit_limit_capital', 'deposit_limit_non_capital',
    'max_loan_capital', 'max_loan_non_capital',
    'ltv_limit_pct', 'dsr_limit_pct', 'max_area_sqm',
    'credit_score_min', 'age_min', 'age_max',
    'income_limit_single', 'income_limit_couple', 'net_asset_limit',
]
for col in num_cols:
    if col in catalog.columns:
        catalog[col] = pd.to_numeric(catalog[col], errors='coerce')

catalog = catalog.dropna(subset=['base_rate_min', 'base_rate_max']).reset_index(drop=True)

gov_cnt  = catalog['product_id'].str.startswith('A').sum()
bank_cnt = len(catalog) - gov_cnt
print(f'고객 프로필   : {len(profiles):>4d}명')
print(f'매물/대출 정보: {len(gates):>4d}건')
print(f'상품 카탈로그 : {len(catalog):>4d}개  (정부지원 {gov_cnt}개 + 금융권 {bank_cnt}개)')
print(f'\n사용 가능한 고객 ID (처음 20개):')
print(', '.join(profiles['user_id'].tolist()[:20]))

In [ ]:
# ══════════════════════════════════════════════════════════════
# Cell 3 : Gate·점수·정책 함수 정의
# ══════════════════════════════════════════════════════════════

# ─────────────────────────────── Gate ──────────────────────────
def evaluate_gate(cust, gate, prod):
    """12개 Gate 조건. 반환: {조건명: 0/1, 'pass': bool}"""
    is_cap = (gate['is_capital_area'] == 'Y')

    income_ok = (
        cust['combined_annual_income'] <= prod['income_limit_couple']
        if cust['marital_status'] == '기혼'
        else cust['annual_income'] <= prod['income_limit_single']
    )
    dep_limit  = prod['deposit_limit_capital'] if is_cap else prod['deposit_limit_non_capital']
    allowed    = str(prod['allowed_property_types']).split('|')

    special = True
    if str(prod.get('marriage_within_7y_required', 'N')) == 'Y':
        special = (cust['marital_status'] == '기혼') and (cust['marriage_years'] <= 7)
    if str(prod.get('newborn_within_2y_required', 'N')) == 'Y':
        special = special and (cust['has_newborn_within_2y'] == 'Y')
    if str(prod.get('youth_only', 'N')) == 'Y':
        special = special and (19 <= cust['age'] <= 34)

    r = {
        '연령조건':     int(prod['age_min'] <= cust['age'] <= prod['age_max']),
        '무주택':       int(cust['is_homeless'] == 'Y'),
        '세대주':       int(cust['is_household_head'] == 'Y'),
        '소득조건':     int(bool(income_ok)),
        '순자산조건':   int(cust['net_asset'] <= prod['net_asset_limit']),
        '신용점수':     int(cust['credit_score_kcb'] >= prod['credit_score_min']),
        'DSR기준':      int(gate['dsr_estimated_pct'] <= prod['dsr_limit_pct']),
        '보증금한도':   int(gate['deposit_amount'] <= dep_limit),
        '주택유형':     int(gate['property_type'] in allowed),
        '전용면적':     int(gate['exclusive_area_sqm'] <= prod['max_area_sqm']),
        '중복대출없음': int(cust['has_existing_jeonse_loan'] == 'N'),
        '정보완전성':   int(gate['info_completeness'] == 'Y'),
        '상품별조건':   int(special),
    }
    r['pass'] = all(v == 1 for k, v in r.items() if k != 'pass')
    return r


# ───────────────────────── 고객 유형 분류 ───────────────────────
def classify_customer(cust, gate, catalog):
    """우선순위: 신용취약 > 상환위험 > 한도부족 > 청년 > 신혼·신생아 > 금리민감 > 기본"""
    median_limit = catalog['max_loan_capital'].median()

    if cust['credit_score_kcb'] < 700:
        return 'credit_vulnerable'
    if gate['dsr_estimated_pct'] >= 35:
        return 'high_dsr_risk'
    if gate['loan_amount_requested'] > median_limit * 0.9:
        return 'limit_constrained'
    if (19 <= cust['age'] <= 34) and (cust['is_homeless'] == 'Y') and (cust['is_household_head'] == 'Y'):
        return 'youth'
    if (cust['marital_status'] == '기혼' and cust['marriage_years'] <= 7) or (cust['has_newborn_within_2y'] == 'Y'):
        return 'newlywed_newborn'
    if cust['combined_annual_income'] <= 4000:
        return 'rate_sensitive'
    return 'default'


# ──────────────────────────── 점수 계산 ─────────────────────────
def calc_monthly_payment(loan_man_won, annual_rate_pct, months=24):
    """분할상환(원리금균등) 기준 월 상환액 (만원)"""
    loan = loan_man_won * 10000
    r    = annual_rate_pct / 100 / 12
    if r > 0:
        monthly = loan * r * (1 + r) ** months / ((1 + r) ** months - 1)
    else:
        monthly = loan / months
    return max(1, int(monthly / 10000))


def effective_limit(gate, prod):
    """LTV 캡 + 지역 상한 적용 실질 한도 (만원)"""
    is_cap   = (gate['is_capital_area'] == 'Y')
    max_loan = prod['max_loan_capital'] if is_cap else prod['max_loan_non_capital']
    ltv_cap  = gate['deposit_amount'] * prod['ltv_limit_pct'] / 100
    return int(min(max_loan, ltv_cap))


def score_product(cust, gate, prod, rate_range, profile_key):
    """JeonseLoanRecScore 계산"""
    w        = WEIGHT_PROFILES[profile_key]
    rep_rate = (prod['base_rate_min'] + prod['base_rate_max']) / 2

    # 1. ApprovalScore
    credit_m = max(0.0,
        (cust['credit_score_kcb'] - prod['credit_score_min']) /
        max(1, 1000 - prod['credit_score_min'])) * 100
    inc_lim  = prod['income_limit_couple'] if cust['marital_status'] == '기혼' \
               else prod['income_limit_single']
    income_m = max(0.0, (inc_lim - cust['combined_annual_income']) /
                   max(1, inc_lim)) * 100  if inc_lim < 99999 else 100.0
    approval = 0.6 * credit_m + 0.4 * income_m

    # 2. InterestBenefitScore (후보군 내 정규화)
    lo, hi = rate_range
    interest = max(0.0, (hi - rep_rate) / (hi - lo) * 100) if hi > lo else 50.0

    # 3. LimitCoverageScore
    eff_lim = effective_limit(gate, prod)
    limit   = min(100.0, eff_lim / max(1, gate['loan_amount_requested']) * 100)

    # 4. AffordabilityScore
    afford  = max(0.0, (prod['dsr_limit_pct'] - gate['dsr_estimated_pct']) /
                  prod['dsr_limit_pct'] * 100)

    # 5. GuaranteeStabilityScore
    guar    = float(GUARANTEE_SCORE.get(str(prod['guarantee_agency']), 60))

    # 6. PreferenceMatchScore
    pref_map = {'고정': '고정금리', '변동': '변동금리', '혼합': '혼합금리'}
    pref_key = pref_map.get(str(gate.get('interest_rate_type', '')), '')
    notes    = str(prod.get('notes', ''))
    pref     = 100.0 if (pref_key and pref_key in notes) else (0.0 if pref_key else 50.0)

    # RiskPenalty
    pen = 0.0
    if 30 <= gate['dsr_estimated_pct'] < 40:               pen += 10
    if 600 <= cust['credit_score_kcb'] < 700:              pen += 15
    if gate['deposit_to_market_ratio_pct'] > 90:           pen += 20
    if gate['senior_plus_deposit_to_market_pct'] > 80:     pen += 10

    score = (w[0]*approval + w[1]*interest + w[2]*limit +
             w[3]*afford   + w[4]*guar     + w[5]*pref  - pen)
    return max(0.0, round(score, 2))


# ────────────────────── 리스크 정책 적용 ────────────────────────
def _extract_bank(product_name: str) -> str:
    """상품명에서 기관명 추출. '주식회사 XX은행 ...' 형태 처리."""
    parts = str(product_name).split()
    if not parts:
        return product_name
    # "주식회사 하나은행 ..." → "하나은행"
    if parts[0] == '주식회사' and len(parts) > 1:
        return parts[1]
    return parts[0]


def apply_policy(raw_ranked, cust, gate, passed_map):
    """
    raw_ranked  : [(score, prod), ...] 점수 내림차순
    passed_map  : {product_id: (score, prod)} Gate 통과 전 상품 (정책 우선편입용)
    반환        : (final_top5, exclusion_log)
      exclusion_log : [(score, prod, reason)]  점수 높지만 제외된 상품

    적용 정책 (순서대로)
    ① 고금리 배제    : base_rate_min > 10% 상품 제외
    ② 기관 중복 제한 : 동일 금융기관 최대 2개 (3번째부터 제외)
    ③ 청년 우대 강제  : 만19~34 무주택 세대주 → A2 Gate 통과 시 최종 Top5 강제 포함
    ④ 신혼·신생아 강제: 혼인 7년 이내 → A3, 2년 내 출산 → A4 강제 포함
    """
    bank_count   = {}
    policy_ok    = []
    policy_excl  = []

    for score, prod in raw_ranked:
        reason = None
        bank   = _extract_bank(prod['product_name'])

        # ① 고금리 배제
        if prod['base_rate_min'] > 10.0:
            reason = f"고금리 상품 배제 (최저금리 {prod['base_rate_min']:.1f}% > 10.0%)"
        # ② 동일 기관 중복 제한 (고금리 제외 상품은 기관 카운트에서 제외)
        elif bank_count.get(bank, 0) >= 2:
            reason = f"동일 금융기관 집중 배제 ({bank}, 상위 2개 초과)"

        if reason:
            policy_excl.append((score, prod, reason))
        else:
            bank_count[bank] = bank_count.get(bank, 0) + 1
            policy_ok.append((score, prod))

    # 정책 통과 상품 중 상위 5개 → 최종 후보
    final    = list(policy_ok[:5])
    excl_log = []

    def _final_ids():
        return {str(p['product_id']) for _, p in final}

    def _force_include(pid, label):
        """pid 가 Gate 통과했고 final에 없으면 강제 삽입 (최하위 상품 제거)"""
        if pid in _final_ids():
            return
        item = passed_map.get(pid)
        if item is None:
            return  # Gate 미통과
        if len(final) >= 5:
            displaced_score, displaced_prod = final[-1]
            final.pop()
            excl_log.append((displaced_score, displaced_prod,
                             f"{label} 강제 포함으로 순위 밀림"))
        fi_score, fi_prod = item
        final.append((fi_score, fi_prod))
        final.sort(key=lambda x: -x[0])

    # ③ 청년 우대
    if (19 <= cust['age'] <= 34 and
            cust['is_homeless'] == 'Y' and
            cust['is_household_head'] == 'Y'):
        _force_include('A2', '청년 우대상품(A2 청년전용 버팀목)')

    # ④ 신혼·신생아 우대
    if cust['marital_status'] == '기혼' and cust['marriage_years'] <= 7:
        _force_include('A3', '신혼부부 우대상품(A3)')
    if cust['has_newborn_within_2y'] == 'Y':
        _force_include('A4', '신생아특례 우대상품(A4)')

    # 정책 탈락 중 raw top5에 있었던 상품 → excl_log 추가
    raw_top5_ids = {str(p['product_id']) for _, p in raw_ranked[:5]}
    final_ids    = _final_ids()
    for score, prod, reason in policy_excl:
        if str(prod['product_id']) in raw_top5_ids and str(prod['product_id']) not in final_ids:
            excl_log.append((score, prod, reason))

    excl_log.sort(key=lambda x: -x[0])
    return final, excl_log


print('✓ Cell 3 완료  (Gate·점수·정책 함수)')

In [ ]:
# ══════════════════════════════════════════════════════════════
# Cell 4 : ★ 고객 ID 선택  ← 여기만 바꾸세요
# ══════════════════════════════════════════════════════════════
USER_ID = 'U0004'   # ← 원하는 고객 ID (예: 'U0042', 'U0100')
print(f'선택 고객 → {USER_ID}')

In [ ]:
# ══════════════════════════════════════════════════════════════
# Cell 5 : 추천 실행
# ══════════════════════════════════════════════════════════════
from collections import Counter

def pname(prod, width=28):
    """상품명 고정 폭 포맷 (너무 길면 축약)"""
    nm = str(prod['product_name'])
    if len(nm) > width:
        nm = nm[:width - 1] + '…'
    return nm.ljust(width)


def recommend(user_id, top_n=5):
    SEP  = '─' * 72
    SEP2 = '═' * 72

    # ── 데이터 조회
    cust_rows = profiles[profiles['user_id'] == user_id]
    gate_rows = gates[gates['user_id'] == user_id]

    if cust_rows.empty:
        print(f'❌ 고객 ID "{user_id}" 를 찾을 수 없습니다.')
        return
    if gate_rows.empty:
        print(f'❌ "{user_id}" 의 매물 정보가 없습니다.')
        return

    cust = cust_rows.iloc[0]
    gate = gate_rows.iloc[0]

    profile_key = classify_customer(cust, gate, catalog)
    is_cap      = (gate['is_capital_area'] == 'Y')
    loan_req    = int(gate['loan_amount_requested'])

    # ── Gate 평가
    passed_list = []
    gate_fail   = []

    for _, prod in catalog.iterrows():
        g = evaluate_gate(cust, gate, prod)
        if g['pass']:
            passed_list.append(prod)
        else:
            fail_keys = [k for k, v in g.items() if k != 'pass' and v == 0]
            gate_fail.append((prod, fail_keys))

    # ── Gate 전부 탈락 시 진단 출력
    if not passed_list:
        print(SEP2)
        print(f'  전세대출 추천  |  고객: {user_id}')
        print(SEP2)
        print(f'  나이 {cust["age"]}세  |  신용점수 {int(cust["credit_score_kcb"])}점  '
              f'|  예상 DSR {gate["dsr_estimated_pct"]:.1f}%  '
              f'|  부부합산 소득 {int(cust["combined_annual_income"]):,}만원')
        print(f'  is_homeless={cust["is_homeless"]}  '
              f'is_household_head={cust["is_household_head"]}  '
              f'has_existing_jeonse_loan={cust["has_existing_jeonse_loan"]}')
        print()
        print('❌ 어떤 상품도 Gate 를 통과하지 못했습니다.')
        print()

        # 조건별 탈락 횟수 집계
        fail_counter = Counter()
        for _, fail_keys in gate_fail:
            for k in fail_keys:
                fail_counter[k] += 1

        total = len(catalog)
        detail_map = {
            '무주택':       f'is_homeless={cust["is_homeless"]} → Y 필요 (무주택 세대원)',
            '세대주':       f'is_household_head={cust["is_household_head"]} → Y 필요 (세대주/예비세대주)',
            '중복대출없음': f'has_existing_jeonse_loan={cust["has_existing_jeonse_loan"]} → N 필요 (중복 대출 불가)',
            '신용점수':     f'신용점수 {int(cust["credit_score_kcb"])}점 → 상품 최저점 미달',
            '소득조건':     f'부부합산 소득 {int(cust["combined_annual_income"]):,}만원 → 상품별 상한 초과',
            '순자산조건':   f'순자산 {int(cust["net_asset"]):,}만원 → 상품별 상한 초과',
            '보증금한도':   f'보증금 {int(gate["deposit_amount"]):,}만원 → 상품별 한도 초과',
            '전용면적':     f'전용면적 {gate["exclusive_area_sqm"]}㎡ → 상품별 상한 초과',
            'DSR기준':      f'예상 DSR {gate["dsr_estimated_pct"]}% → 상품별 한도 초과',
            '주택유형':     f'주택유형 {gate["property_type"]} → 허용 유형 불일치',
            '정보완전성':   f'info_completeness={gate["info_completeness"]} → Y 필요',
            '상품별조건':   f'혼인/출산/청년 전용 조건 미충족',
            '연령조건':     f'나이 {cust["age"]}세 → 상품별 연령 범위 외',
        }

        print(f'  [Gate 탈락 원인 분석]  (전체 {total}개 상품 모두 탈락)')
        print(f'  {"탈락 조건":<14}  {"차단 수":>7}  {"원인 상세"}')
        print('  ' + '─' * 65)
        for cond, cnt in fail_counter.most_common():
            block_all = ' ← 전 상품 차단' if cnt == total else ''
            detail    = detail_map.get(cond, '')
            print(f'  {cond:<14}  {cnt:>4}/{total}  {detail}{block_all}')

        # Gate 통과 가능 고객 안내
        print()
        print('  💡 Gate 통과 가능한 고객 ID 예시 (처음 10개):')
        passing = []
        for uid in profiles['user_id']:
            if len(passing) >= 10:
                break
            c = profiles[profiles['user_id'] == uid].iloc[0]
            g = gates[gates['user_id'] == uid].iloc[0]
            for _, p in catalog.iterrows():
                gr = evaluate_gate(c, g, p)
                if gr['pass']:
                    passing.append(uid)
                    break
        print(f'  {", ".join(passing)}')
        print(SEP)
        return

    # ── 점수 계산
    rep_rates  = [(p['base_rate_min'] + p['base_rate_max']) / 2 for p in passed_list]
    rate_range = (min(rep_rates), max(rep_rates))

    raw_ranked = sorted(
        [(score_product(cust, gate, p, rate_range, profile_key), p) for p in passed_list],
        key=lambda x: -x[0]
    )

    passed_map = {str(p['product_id']): (s, p) for s, p in raw_ranked}

    # ── 헤더
    print(SEP2)
    print(f'  전세대출 Top {top_n} 추천  |  고객: {user_id}')
    print(SEP2)
    w = WEIGHT_PROFILES[profile_key]
    print(f'  고객 유형 : {WEIGHT_LABELS[profile_key]}')
    print(f'  나이 {cust["age"]}세  |  신용점수 {int(cust["credit_score_kcb"])}점  '
          f'|  예상 DSR {gate["dsr_estimated_pct"]:.1f}%  '
          f'|  부부합산 소득 {int(cust["combined_annual_income"]):,}만원')
    print(f'  희망 대출 {loan_req:,}만원  |  전세 보증금 {int(gate["deposit_amount"]):,}만원  '
          f'|  주택 유형 {gate["property_type"]}  '
          f'|  {"수도권" if is_cap else "비수도권"}')
    print(f'  가중치   : 승인 {w[0]:.0%} | 금리 {w[1]:.0%} | 한도 {w[2]:.0%} '
          f'| 상환 {w[3]:.0%} | 보증 {w[4]:.0%} | 우대 {w[5]:.0%}')
    print(f'  Gate 통과: {len(passed_list)}개 / {len(catalog)}개 상품')

    # ── 1단계: 점수 기준 Top N
    print(f'\n{SEP}')
    print('  [1단계] 점수 기준 Top 5  (리스크 정책 적용 전)')
    print(SEP)
    print(f'  {"순위":>2}  {"점수":>5}  {"ID":>5}  {"상품명":<28}  {"금리(최저)":>9}  {"실질한도":>10}  {"보증":>5}')
    print('  ' + '─' * 70)

    for rank, (score, prod) in enumerate(raw_ranked[:top_n], 1):
        eff = effective_limit(gate, prod)
        print(
            f'  {rank:>2}위  {score:>5.1f}  '
            f'{str(prod["product_id"]):>5}  '
            f'{pname(prod)}  '
            f'{prod["base_rate_min"]:>7.2f}%  '
            f'{eff:>9,}만원  '
            f'{str(prod["guarantee_agency"]):>5}'
        )

    # ── 2단계: 정책 적용 후 최종 Top N
    final_top5, excl_log = apply_policy(raw_ranked, cust, gate, passed_map)

    print(f'\n{SEP}')
    print('  [2단계] 리스크 정책 적용 → 최종 추천 Top 5')
    print(SEP)
    print(f'  {"순위":>2}  {"점수":>5}  {"ID":>5}  {"상품명":<28}  '
          f'{"금리(최저)":>9}  {"실질한도":>10}  {"월상환(24M)":>10}  {"보증":>5}')
    print('  ' + '─' * 70)

    for rank, (score, prod) in enumerate(final_top5, 1):
        eff      = effective_limit(gate, prod)
        act_loan = min(loan_req, eff)
        monthly  = calc_monthly_payment(act_loan, prod['base_rate_min'])
        print(
            f'  {rank:>2}위  {score:>5.1f}  '
            f'{str(prod["product_id"]):>5}  '
            f'{pname(prod)}  '
            f'{prod["base_rate_min"]:>7.2f}%  '
            f'{eff:>9,}만원  '
            f'{monthly:>8,}만원  '
            f'{str(prod["guarantee_agency"]):>5}'
        )

    # 적용 정책 표시
    policies = ['② 동일 기관 최대 2개']
    if any(p['base_rate_min'] > 10 for _, p in raw_ranked):
        policies.insert(0, '① 고금리 배제(>10%)')
    if 19 <= cust['age'] <= 34 and cust['is_homeless'] == 'Y':
        policies.append('③ 청년 우대상품 강제 포함')
    if cust['marital_status'] == '기혼' and cust['marriage_years'] <= 7:
        policies.append('④ 신혼부부 상품 강제 포함')
    if cust['has_newborn_within_2y'] == 'Y':
        policies.append('④ 신생아특례 상품 강제 포함')
    print(f'\n  적용 정책: {"  /  ".join(policies)}')

    # ── 제외 상품
    if excl_log:
        print(f'\n{SEP}')
        print('  ✗ 점수 높지만 제외된 상품')
        print(SEP)
        for score, prod, reason in excl_log:
            nm = str(prod['product_name'])
            if len(nm) > 30:
                nm = nm[:29] + '…'
            print(f'  [{score:>5.1f}점]  {str(prod["product_id"]):>5}  {nm:<31}  →  {reason}')
    else:
        print(f'\n  ✓ 1단계 Top 5 와 최종 Top 5 가 동일합니다 (제외 상품 없음)')

    print(f'\n{SEP}')
    print('  * 금리: 상품 기준금리 최저값 기준 (우대금리 미반영)')
    print('  * 실질 한도: LTV 및 지역 상한 동시 적용 / 월상환: 원리금균등 24개월 기준 추정')
    print(SEP)


# ── 실행
recommend(USER_ID, top_n=5)